# Feature-family ablation, uncertainty, and parsimony

**Research question:** Which challenger earns operational promotion under a fixed daily review queue?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Nonlinearity should beat the calibrated linear baseline.
- Added identity or behaviour must clear a material precision gate.
- Scenario economics must not be reported as realized savings.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: fraud-ops-intelligence-lab


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,TransactionAmt,transaction core,train_transaction.TransactionAmt,transaction,Raw amount,current transaction,yes,none,median in baseline/challenger,model,NaN,low,amount may proxy customer/channel mix,temporal OOT metrics
1,ProductCD/card/address/email/device core,transaction core,train_transaction,transaction,Selected numeric/categorical source fields,current transaction,yes,development,median/most-frequent then ordinal encoding,model,NaN,medium: anonymous semantics,possible geography/device proxies,feature-family ablation
2,has_identity,identity,train_identity join,transaction,1 when identity row exists,current transaction,yes,none,zero,research challenger,NaN,medium: availability process,channel/coverage proxy,identity ablation and uncertainty
3,DeviceType/id_31,identity,train_identity,transaction,Categorical identity values,current transaction,yes,development,most-frequent/unknown encoding,research challenger,NaN,medium,device/browser proxy,parsimony gate; not selected
4,entity_prior_tx_count,past behaviour,proxy entity transaction history,transaction,Count transactions in strictly earlier timesta...,strictly before TransactionDT,yes,none,zero,research challenger,NaN,low after batch-safe grouping,depends on unverified entity proxy,same-timestamp unit test
5,entity_prior_amount_mean,past behaviour,proxy entity transaction history,transaction,Mean amount over strictly earlier timestamp gr...,strictly before TransactionDT,yes,none,missing for new entity,research challenger,NaN,low,depends on unverified entity proxy,same-timestamp unit test
6,entity_seconds_since_previous,past behaviour,proxy entity transaction history,transaction,Elapsed seconds since previous timestamp group,strictly before TransactionDT,yes,none,missing for new entity,research challenger,NaN,low,depends on entity resolution,same-timestamp unit test
7,amount_to_entity_prior_mean,past behaviour,current amount and past-only mean,transaction,TransactionAmt / entity_prior_amount_mean,past history plus current amount,yes,none,missing when no prior mean,research challenger,NaN,low,depends on entity resolution,ablation
8,entity_is_new,past behaviour,entity_prior_tx_count,transaction,1 when no strictly prior transaction exists,strictly before TransactionDT,yes,none,zero,research challenger,NaN,low,newness depends on observation window,ablation
9,entity_key,entity proxy,card/address/email components,transaction,Normalized string concatenation,current transaction,proxy only,none,explicit missing token,queue deduplication,NaN,medium: merges/splits are unverified,identity resolution risk,proxy sensitivity required before production


,feature_family,model_role,features
0,entity proxy,queue deduplication,1
1,identity,research challenger,2
2,past behaviour,research challenger,5
3,transaction core,model,2


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,TransactionAmt,current transaction,yes,none,low,temporal OOT metrics
1,ProductCD/card/address/email/device core,current transaction,yes,development,medium: anonymous semantics,feature-family ablation
2,has_identity,current transaction,yes,none,medium: availability process,identity ablation and uncertainty
3,DeviceType/id_31,current transaction,yes,development,medium,parsimony gate; not selected
4,entity_prior_tx_count,strictly before TransactionDT,yes,none,low after batch-safe grouping,same-timestamp unit test
5,entity_prior_amount_mean,strictly before TransactionDT,yes,none,low,same-timestamp unit test
6,entity_seconds_since_previous,strictly before TransactionDT,yes,none,low,same-timestamp unit test
7,amount_to_entity_prior_mean,past history plus current amount,yes,none,low,ablation
8,entity_is_new,strictly before TransactionDT,yes,none,low,ablation
9,entity_key,current transaction,proxy only,none,medium: merges/splits are unverified,proxy sensitivity required before production


## Committed analytical evidence

In [4]:
evidence_files = ["reports/challenger_model_metrics.csv", "reports/model_comparison_uncertainty.csv", "reports/challenger_decision.json"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/challenger_model_metrics.csv`

,model,partition,rows,fraud_rate,average_precision,roc_auc,brier_score,feature_count,boosting_iterations
0,gbm_transaction,policy,59054,0.034155,0.398293,0.881363,0.025426,46,140
1,gbm_transaction,oot,88581,0.034804,0.460869,0.880345,0.024140,46,140
2,gbm_plus_identity,policy,59054,0.034155,0.404105,0.883387,0.025257,49,140
3,gbm_plus_identity,oot,88581,0.034804,0.459692,0.882719,0.024192,49,140
4,gbm_plus_behaviour,policy,59054,0.034155,0.401026,0.879540,0.025442,54,140
5,gbm_plus_behaviour,oot,88581,0.034804,0.455037,0.878099,0.024354,54,140


### `reports/model_comparison_uncertainty.csv`

,partition,candidate,reference,metric,days,mean_difference,ci_low_95,ci_high_95,probability_candidate_better
0,policy,gbm_transaction,linear,precision_at_k,23,0.191739,0.160435,0.220870,1.0000
1,policy,gbm_transaction,gbm_plus_identity,precision_at_k,23,-0.003043,-0.009565,0.003043,0.1764
2,policy,gbm_transaction,gbm_plus_behaviour,precision_at_k,23,0.000435,-0.004783,0.005228,0.5620
3,oot,gbm_transaction,linear,precision_at_k,31,0.206774,0.186774,0.227097,1.0000
4,oot,gbm_transaction,gbm_plus_identity,precision_at_k,31,0.001290,-0.006774,0.009355,0.6188
5,oot,gbm_transaction,gbm_plus_behaviour,precision_at_k,31,0.002258,-0.004839,0.009355,0.7371


### `reports/challenger_decision.json`

{'selected_on_policy_window': 'gbm_transaction',
 'selection_reason': 'parsimony_gate: best added-feature gain 0.0030 is below 0.0100',
 'parsimony_gate_absolute_precision_gain': 0.01,
 'selection_metric': 'mean daily precision at 100 entity-deduplicated alerts',
 'oot_average_precision': 0.4608689910284057,
 'oot_roc_auc': 0.8803450148517206,
 'oot_brier_score': 0.024139825940593107,
 'oot_mean_daily_precision_at_100': 0.34838709677419355,
 'oot_mean_daily_recall_at_100': 0.35804712214255374,
 'oot_mean_daily_exposure_capture_at_100': 0.3243731192908206,
 'oot_legitimate_exposure_interrupted_at_100': 356307.918,
 'precision_uplift_vs_linear': 1.4601366742596817,
 'average_precision_uplift_vs_linear': 3.208205901415573}

## Decision

**Select the transaction-only GBM; added feature families do not clear the parsimony gate.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.